# Safety, Bias এবং Toxicity Mitigation

দুইটি স্বয়ংসম্পূর্ণ খেলনা প্রদর্শন, সাথে দুটিকে একসূত্রে বাঁধা একটি ছোট guardrail sweep:

1. **Part 1 -- একটি TEMPLATE-BASED BIAS PROBE।** একটি বাক্য template-কে বিভিন্ন demographic term দিয়ে পূরণ করা হয় (pronoun "he" / "she" / "they", একটি protected attribute-এর প্রতিনিধি হিসেবে), কয়েকটি পেশার সাথে মিলিয়ে, এবং এমন একটি scoring function দিয়ে score করা হয় যার ভেতরে একটি pronoun group-এর দিকে একটি **জানা, ইচ্ছাকৃতভাবে ঢোকানো bias** রয়েছে (যা সেই ধরনের ভুয়া, identity-term-সম্পর্কিত bias-এর প্রতিনিধিত্ব করে যা আসল classifier-রা শেখে বলে নথিভুক্ত হয়েছে, যেমন Dixon et al., 2018)। অন্যথায় সমতুল্য prompt-এ group-গুলোর মধ্যে score distribution তুলনা করলে সেই ঢোকানো bias প্রায় হুবহু পুনরুদ্ধার হয় -- প্রমাণ যে probe পদ্ধতিটি আসলেই এমন একটি bias শনাক্ত করে যা, নির্মাণগতভাবে, আমরা জানি সত্যিই আছে।
2. **Part 2 -- শূন্য থেকে একটি TOXICITY CLASSIFIER।** একটি logistic regression (bag-of-words feature, PyTorch দিয়ে train করা) toxic / non-toxic বাক্যাংশের একটি ক্ষুদ্র, হাতে-label করা খেলনা dataset-এ, এমন একটি held-out set-এ precision/recall দিয়ে মূল্যায়িত যেটিতে classifier কখনো train হয়নি।
3. **Part 3 --** train করা classifier-টিকে একটি **INFERENCE-TIME GUARDRAIL** হিসেবে ব্যবহার, candidate output-এর একটি নতুন batch-এ প্রয়োগ করে দেখানো যে একটি "user-কে দেখানোর আগে block করা" filter বাস্তবে কেমন দেখায়, এবং এর মাপা catch rate -- একটি training-data toxicity filter-এর একই মূল ধারণা, শুধু data-curation-এর সময়ের বদলে generation-এর সময় প্রয়োগ করা।

**Runtime:** CPU-তে কয়েক সেকেন্ড।

**চালানোর নিয়ম:** উপর থেকে নিচ পর্যন্ত সব cell ক্রমানুসারে চালান। Part 2-এর train করা `model` ও `vocab` Part 3-এ লাগে এবং সেগুলো `main()`-এর ভেতরে তৈরি হয়, তাই তিনটি part-ই শেষ cell-এ `main()` call করে একসাথে চালানো হয়; আগের cell-গুলো শুধু সংজ্ঞা দেয়।

In [ ]:
import re

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)

## 1. Part 1: Template-based bias probe

পদ্ধতি: template এবং "বৈধ" বিষয়বস্তু (অভিজ্ঞতার বছর, পেশা) স্থির ও সব group-এ **একইভাবে বণ্টিত** রাখা হয়, শুধু demographic term বদলানো হয়, এবং দেখা হয় score distribution তবুও group অনুযায়ী ভিন্ন হয় কিনা। `toy_biased_scorer`-এর বৈধ signal আসে শুধু অভিজ্ঞতার বছর থেকে, কিন্তু এর উপর pronoun-ভিত্তিক একটি স্থির bonus/penalty (`TRUE_INJECTED_BIAS`) যোগ করা হয়। `run_bias_probe()` প্রতি group-এ 300টি বাক্য score করে, group-ভিত্তিক গড় ও std দেখায়, এবং probe-এর শনাক্ত gap-কে আসল ঢোকানো gap-এর সাথে তুলনা করে।

In [ ]:
# ===========================================================================
# PART 1: TEMPLATE-BASED BIAS PROBE
# ===========================================================================
#
# পদ্ধতি: template এবং "বৈধ" বিষয়বস্তু (অভিজ্ঞতার বছর, পেশা) স্থির ও সব
# group-এ একইভাবে বণ্টিত রাখা, শুধু demographic term বদলানো, এবং দেখা score
# distribution তবুও group অনুযায়ী ভিন্ন হয় কিনা। মূল scorer যদি ন্যায্য হতো,
# বছর/পেশা হিসাবে নেওয়ার পর score-এর উপর group-এর কোনো প্রভাব থাকত না --
# অবশিষ্ট যেকোনো gap কেবল demographic term-এর কারণেই।

OCCUPATIONS = ["engineer", "nurse", "surgeon", "teacher", "programmer", "receptionist"]
PRONOUNS = ["he", "she", "they"]

TEMPLATE = "{pronoun} has {years} years of experience as a {occupation} and is highly skilled."

# খেলনা scorer-এ ইচ্ছাকৃতভাবে ঢোকানো bias -- probe-এর কাছে অজানা, আমাদের
# কাছে জানা, যাতে যাচাই করা যায় probe সেটি পুনরুদ্ধার করে কিনা।
TRUE_INJECTED_BIAS = {"he": 0.40, "she": 0.00, "they": -0.20}


def toy_biased_scorer(sentence):
    """একটি ইচ্ছাকৃতভাবে biased খেলনা 'quality/competence' scorer। বৈধ
    signal আসে শুধু অভিজ্ঞতার বছর থেকে (text থেকে বের করা); তার উপর,
    বাক্যে কোন pronoun আছে শুধু তার ভিত্তিতে একটি স্থির bonus/penalty যোগ
    করা হয় -- ঠিক সেই ধরনের ভুয়া, identity-term-সংযুক্ত shortcut যা একটি
    আসল learned classifier biased training data থেকে তুলে নিতে পারে, আসল
    quality-র সাথে যার কোনো সম্পর্ক নেই।"""
    years = int(re.search(r"(\d+) years", sentence).group(1))
    pronoun = sentence.split()[0].lower()
    legitimate_signal = 0.1 * years                     # একমাত্র signal যা গুরুত্বপূর্ণ হওয়া উচিত
    injected_bias = TRUE_INJECTED_BIAS[pronoun]           # যে bias থাকা উচিত নয়
    noise = np.random.normal(0, 0.3)
    return legitimate_signal + injected_bias + noise


def run_bias_probe():
    print("=" * 70)
    print("PART 1: TEMPLATE-BASED BIAS PROBE")
    print("=" * 70)
    print(f"Template: {TEMPLATE!r}")
    print(f"Demographic terms probed: {PRONOUNS}")
    print(f"(Unknown to the probe) bias actually injected into the scorer: {TRUE_INJECTED_BIAS}\n")

    scores_by_group = {p: [] for p in PRONOUNS}
    SAMPLES_PER_GROUP = 300
    for pronoun in PRONOUNS:
        for _ in range(SAMPLES_PER_GROUP):
            occupation = np.random.choice(OCCUPATIONS)
            years = np.random.randint(1, 16)   # প্রতিটি group-এর জন্য একই distribution -- কোনো confound নেই
            sentence = TEMPLATE.format(pronoun=pronoun.capitalize(), years=years, occupation=occupation)
            scores_by_group[pronoun].append(toy_biased_scorer(sentence))

    print(f"Generated {SAMPLES_PER_GROUP} scored sentences per group ({len(PRONOUNS) * SAMPLES_PER_GROUP} total),")
    print("with years-of-experience and occupation drawn from the SAME distribution for every group,")
    print("so any remaining score difference between groups cannot be explained by those legitimate")
    print("factors -- only by the demographic term itself.\n")

    means = {p: np.mean(scores_by_group[p]) for p in PRONOUNS}
    stds = {p: np.std(scores_by_group[p]) for p in PRONOUNS}
    print(f"{'group':>8}{'mean score':>14}{'std dev':>12}{'n':>8}")
    for p in PRONOUNS:
        print(f"{p:>8}{means[p]:>14.3f}{stds[p]:>12.3f}{SAMPLES_PER_GROUP:>8}")

    baseline = "she"   # gap report করার জন্য একটি যথেচ্ছ reference group
    print(f"\nMeasured gap in mean score relative to '{baseline}' (the probe's finding):")
    detected_gaps = {}
    for p in PRONOUNS:
        gap = means[p] - means[baseline]
        detected_gaps[p] = gap
        true_gap = TRUE_INJECTED_BIAS[p] - TRUE_INJECTED_BIAS[baseline]
        print(f"  '{p}' vs '{baseline}': detected gap = {gap:+.3f}   "
              f"(true injected gap = {true_gap:+.3f}, error = {abs(gap - true_gap):.3f})")

    max_error = max(abs(detected_gaps[p] - (TRUE_INJECTED_BIAS[p] - TRUE_INJECTED_BIAS[baseline]))
                     for p in PRONOUNS)
    print(f"\n-> Despite years-of-experience and occupation being drawn identically across all three")
    print(f"   groups, the probe still detects a systematic score gap that tracks the injected bias")
    print(f"   closely (largest error across groups: {max_error:.3f} score points, from sampling noise")
    print(f"   alone). This is exactly the template-probe methodology real fairness audits use: hold")
    print(f"   everything legitimate fixed, vary only the demographic term, and treat a resulting gap")
    print(f"   as evidence of bias in the model or scorer being probed -- here we control the ground")
    print(f"   truth and confirm the methodology correctly recovers a bias we know is really there.")

## 2. Part 2: শূন্য থেকে একটি toxicity classifier (logistic regression, bag-of-words)

toxic ও clean বাক্যাংশের ছোট train/test set, একটি সাধারণ tokenizer, binary bag-of-words feature, এবং একটি single `nn.Linear` layer-এর logistic regression classifier। `train_toxicity_classifier()` BCE loss দিয়ে 300 ধাপ train করে `(model, vocab)` ফেরত দেয়; `evaluate_classifier()` held-out set-এ confusion matrix, precision, recall ও F1 দেখায়।

In [ ]:
# ===========================================================================
# PART 2: শূন্য থেকে একটি TOXICITY CLASSIFIER (logistic regression, BoW)
# ===========================================================================

TOXIC_TRAIN = [
    "you are so stupid and worthless",
    "shut up you idiot nobody likes you",
    "i hate you, you are pathetic and useless",
    "you are a disgusting moron and should feel ashamed",
    "get lost you worthless piece of trash",
    "everyone thinks you are an idiot and a loser",
    "you are garbage and do not deserve respect",
    "i will destroy you, you pathetic fool",
]
CLEAN_TRAIN = [
    "thank you so much for your help today",
    "i really appreciate your hard work on this project",
    "the weather looks lovely this afternoon",
    "could you please send me the report by friday",
    "i think this recipe would taste great with more garlic",
    "the meeting has been rescheduled to next tuesday",
    "your presentation was well organized and clear",
    "let us know if you have any questions about the plan",
]

TOXIC_TEST = [
    "you are such a worthless idiot",
    "shut your mouth you disgusting fool",
    "nobody wants you here you loser",
    "you are pathetic and should feel ashamed",
    "get out of here you disgusting piece of garbage",
]
CLEAN_TEST = [
    "the new library opens next monday",
    "i would love to grab coffee sometime this week",
    "your feedback on the report was really helpful",
    "please review the attached document when you can",
    "the garden looks beautiful after the rain",
]


def tokenize(text):
    return re.findall(r"[a-z']+", text.lower())


def build_vocab(texts):
    vocab = sorted({tok for text in texts for tok in tokenize(text)})
    return {tok: i for i, tok in enumerate(vocab)}


def to_bow(text, vocab):
    vec = np.zeros(len(vocab), dtype=np.float32)
    for tok in tokenize(text):
        if tok in vocab:               # vocabulary-র বাইরের শব্দ এই model-এর কাছে সম্পূর্ণ অদৃশ্য
            vec[vocab[tok]] = 1.0      # binary bag-of-words (শব্দ আছে / নেই)
    return vec


class ToxicityClassifier(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.linear = nn.Linear(vocab_size, 1)

    def forward(self, x):
        return self.linear(x).squeeze(-1)   # raw logit; probability পেতে sigmoid প্রয়োগ করুন


def train_toxicity_classifier():
    vocab = build_vocab(TOXIC_TRAIN + CLEAN_TRAIN)
    train_texts = TOXIC_TRAIN + CLEAN_TRAIN
    train_labels = [1.0] * len(TOXIC_TRAIN) + [0.0] * len(CLEAN_TRAIN)
    X_train = torch.tensor(np.stack([to_bow(t, vocab) for t in train_texts]))
    y_train = torch.tensor(train_labels)

    model = ToxicityClassifier(len(vocab))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

    for step in range(1, 301):
        optimizer.zero_grad()
        logits = model(X_train)
        loss = F.binary_cross_entropy_with_logits(logits, y_train)
        loss.backward()
        optimizer.step()
        if step % 100 == 0 or step == 1:
            print(f"  train step {step:4d}   BCE loss = {loss.item():.4f}")

    return model, vocab


def evaluate_classifier(model, vocab, threshold=0.5):
    test_texts = TOXIC_TEST + CLEAN_TEST
    test_labels = [1] * len(TOXIC_TEST) + [0] * len(CLEAN_TEST)
    X_test = torch.tensor(np.stack([to_bow(t, vocab) for t in test_texts]))
    with torch.no_grad():
        probs = torch.sigmoid(model(X_test)).numpy()
    preds = (probs >= threshold).astype(int)

    tp = int(np.sum((preds == 1) & (np.array(test_labels) == 1)))
    fp = int(np.sum((preds == 1) & (np.array(test_labels) == 0)))
    fn = int(np.sum((preds == 0) & (np.array(test_labels) == 1)))
    tn = int(np.sum((preds == 0) & (np.array(test_labels) == 0)))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    print(f"\nHeld-out set: {len(TOXIC_TEST)} toxic + {len(CLEAN_TEST)} clean phrases, "
          f"none seen during training.")
    print(f"{'text':>52}{'true':>8}{'p(toxic)':>10}{'pred':>8}")
    for text, label, prob, pred in zip(test_texts, test_labels, probs, preds):
        print(f"{text:>52}{label:>8}{prob:>10.3f}{pred:>8}")

    print(f"\nConfusion matrix on held-out set: TP={tp}  FP={fp}  FN={fn}  TN={tn}")
    print(f"Precision = TP/(TP+FP) = {precision:.3f}")
    print(f"Recall    = TP/(TP+FN) = {recall:.3f}")
    print(f"F1        = {f1:.3f}")
    return precision, recall, f1

## 3. Part 3: একই classifier একটি inference-time guardrail হিসেবে

হুবহু একই train করা model, pipeline-এর একটি ভিন্ন ধাপে প্রয়োগ করা: pretraining-এর আগে training data filter করার বদলে, এটি সদ্য generated candidate output-এর একটি batch user-কে দেখানোর ঠিক আগে screen করে -- একটি হালকা "red-team sweep" যা deploy করার আগে কী ফসকে যায় আর কী ধরা পড়ে তা খুঁজে বের করে। `run_guardrail_sweep()` প্রতিটি output-কে BLOCKED/ALLOWED হিসেবে চিহ্নিত করে এবং উদ্দিষ্ট action-এর সাথে কতগুলো মিলেছে তা গোনে।

In [ ]:
# ===========================================================================
# PART 3: একই CLASSIFIER একটি INFERENCE-TIME GUARDRAIL হিসেবে ব্যবহৃত
# ===========================================================================
# হুবহু একই train করা model, একটি ভিন্ন pipeline ধাপে প্রয়োগ করা: pretraining-এর
# আগে TRAINING data filter করার বদলে, এটি সদ্য generated candidate output-এর
# একটি batch user-কে দেখানোর ঠিক আগে screen করে -- একটি হালকা, ইচ্ছাকৃতভাবে
# খোঁজাখুঁজি করা "red-team sweep" যা adversarial-ধাঁচের output-এর একটি batch
# দিয়ে একটি deployed system পরীক্ষা করার অনুকরণ করে, deployment-এর আগেই কী
# ফসকে যায় আর কী ধরা পড়ে তা খুঁজে বের করতে।

DEPLOYMENT_BATCH = [
    ("you are worthless and everyone hates you", True),     # block হওয়া উচিত
    ("thanks again, this was really useful", False),         # allow হওয়া উচিত
    ("shut up, nobody asked for your garbage opinion", True),  # block হওয়া উচিত
    ("the train departs from platform three", False),        # allow হওয়া উচিত
    ("you disgusting idiot, get out of my sight", True),      # block হওয়া উচিত
]


def run_guardrail_sweep(model, vocab, threshold=0.5):
    print("\n" + "=" * 70)
    print("PART 3: THE TRAINED CLASSIFIER AS AN INFERENCE-TIME GUARDRAIL")
    print("=" * 70)
    print("Same model, same weights -- now used to screen a fresh batch of candidate")
    print("outputs before they would ever reach a user (none of these five texts were")
    print("used for training or the held-out evaluation above).\n")

    texts = [t for t, _ in DEPLOYMENT_BATCH]
    should_block = [b for _, b in DEPLOYMENT_BATCH]
    X = torch.tensor(np.stack([to_bow(t, vocab) for t in texts]))
    with torch.no_grad():
        probs = torch.sigmoid(model(X)).numpy()
    blocked = probs >= threshold

    correct = 0
    for text, expect_block, prob, is_blocked in zip(texts, should_block, probs, blocked):
        action = "BLOCKED" if is_blocked else "ALLOWED"
        ok = (is_blocked == expect_block)
        correct += int(ok)
        print(f"  p(toxic)={prob:.3f}  ->  {action:8s}  {'(correct)' if ok else '(MISCLASSIFIED)'}"
              f"   {text!r}")

    print(f"\nGuardrail matched the intended action on {correct}/{len(DEPLOYMENT_BATCH)} "
          f"fresh deployment-style examples.")
    print("-> This is the inference-time-guardrail mitigation stage: a small, fast, dedicated")
    print("   classifier sits between the language model and the user, and can block a response")
    print("   the language model itself was never trained to refuse -- distinct from filtering")
    print("   at pretraining-data time and distinct from steering the policy itself with RLHF/DPO,")
    print("   and useful precisely because it can be updated and audited independently of the")
    print("   (far more expensive to retrain) language model it is guarding.")

## সবগুলো demo একসাথে

`main()` তিনটি part ক্রমানুসারে চালায়: bias probe (Part 1), toxicity classifier train ও মূল্যায়ন (Part 2), এবং সেই একই train করা `model` ও `vocab` দিয়ে guardrail sweep (Part 3)। যেহেতু Part 2-এর ফলাফল `main()`-এর ভেতরেই Part 3-এ পাঠানো হয়, এই notebook-এ সব demo এখানেই একবার চলে — তাই `main()` সরাসরি call করা হয়েছে।

In [ ]:
def main():
    run_bias_probe()

    print("\n" + "=" * 70)
    print("PART 2: A TOXICITY CLASSIFIER FROM SCRATCH (logistic regression, bag-of-words)")
    print("=" * 70)
    print(f"Training on {len(TOXIC_TRAIN)} toxic + {len(CLEAN_TRAIN)} clean labeled toy phrases.\n")
    model, vocab = train_toxicity_classifier()
    evaluate_classifier(model, vocab)

    run_guardrail_sweep(model, vocab)


main()